# 03 - Capa Gold\n\nConstrucción de data marts comerciales para consumo analítico: ventas por mes, tienda, producto, canal y documentos.

# Configuración base

Este notebook está preparado para ejecutarse en Google Colab usando Google Drive personal.

Estructura esperada en Drive:

```text
Proyecto_BigData_Forus/
├── raw/
│   └── venta_tiendas.csv
├── bronze/
├── silver/
├── gold/
└── evidencias/
```


In [1]:
# Instalación de dependencias para Colab
!pip install -q pyspark==3.5.1 delta-spark==3.2.0


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.0/317.0 MB 5.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.5/200.5 kB 8.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dataproc-spark-connect 1.1.0 requires pyspark[connect]~=4.0.0, but you have pyspark 3.5.1 which is incompatible.


In [3]:
from google.colab import drive
drive.mount('/content/drive')

import os
import time
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

RUTA_BASE = "/content/drive/MyDrive/Proyecto_BigData_Forus"

RUTA_RAW = f"{RUTA_BASE}/raw"
RUTA_BRONZE = f"{RUTA_BASE}/bronze"
RUTA_SILVER = f"{RUTA_BASE}/silver"
RUTA_GOLD = f"{RUTA_BASE}/gold"
RUTA_EVIDENCIAS = f"{RUTA_BASE}/evidencias"

ARCHIVO_VENTAS = "venta_tiendas.csv"

for ruta in [RUTA_RAW, RUTA_BRONZE, RUTA_SILVER, RUTA_GOLD, RUTA_EVIDENCIAS]:
    os.makedirs(ruta, exist_ok=True)

builder = (
    SparkSession.builder
    .appName("Forus_Fase2_BigData")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.sql.shuffle.partitions", "8")
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

print("Spark:", spark.version)
print("Ruta base:", RUTA_BASE)


Mounted at /content/drive
Spark: 3.5.1
Ruta base: /content/drive/MyDrive/Proyecto_BigData_Forus


## 1. Lectura desde Silver

In [4]:
df_silver = spark.read.format("delta").load(f"{RUTA_SILVER}/venta_tiendas")

print("Registros Silver:", df_silver.count())
df_silver.printSchema()
df_silver.show(5, truncate=False)


Registros Silver: 2250970
root
 |-- id_canal: integer (nullable = true)
 |-- numero_transaccion: long (nullable = true)
 |-- numero_pos: integer (nullable = true)
 |-- numero_boleta: long (nullable = true)
 |-- fecha_transaccion: string (nullable = true)
 |-- cod_tienda_facturacion: integer (nullable = true)
 |-- tipo_documento: string (nullable = true)
 |-- id_producto: long (nullable = true)
 |-- unidades: integer (nullable = true)
 |-- venta: double (nullable = true)
 |-- costo: double (nullable = true)
 |-- fecha_transaccion_guion: string (nullable = true)
 |-- fecha_timestamp: timestamp (nullable = true)
 |-- fecha_venta: date (nullable = true)
 |-- fecha_venta_texto: string (nullable = true)
 |-- anio: integer (nullable = true)
 |-- mes: integer (nullable = true)
 |-- dia: integer (nullable = true)
 |-- margen: double (nullable = true)
 |-- margen_porcentaje: double (nullable = true)

+--------+------------------+----------+-------------+-------------------------+----------------

In [12]:
# ============================================================
# Evidencia de optimización: comparación sin cache vs con cache
# ============================================================

import time
from pyspark.sql.functions import sum, col

# Consulta base sin cache
inicio = time.time()

resultado_sin_cache = (
    df_silver
    .groupBy("anio", "mes")
    .agg(sum("venta").alias("venta_total"))
)

resultado_sin_cache.count()

tiempo_sin_cache = time.time() - inicio

# Consulta optimizada con cache
df_silver_cache = df_silver.cache()
df_silver_cache.count()

inicio = time.time()

resultado_con_cache = (
    df_silver_cache
    .groupBy("anio", "mes")
    .agg(sum("venta").alias("venta_total"))
)

resultado_con_cache.count()

tiempo_con_cache = time.time() - inicio

mejora_pct = ((tiempo_sin_cache - tiempo_con_cache) / tiempo_sin_cache) * 100

print("Tiempo sin cache:", round(tiempo_sin_cache, 2), "segundos")
print("Tiempo con cache:", round(tiempo_con_cache, 2), "segundos")
print("Mejora porcentual:", round(mejora_pct, 2), "%")

Tiempo sin cache: 0.54 segundos
Tiempo con cache: 0.31 segundos
Mejora porcentual: 43.08 %


In [13]:
# ============================================================
# Tabla resumen de optimización
# ============================================================

df_optimizacion = spark.createDataFrame([
    ("Consulta ventas por año/mes", "Sin cache", round(tiempo_sin_cache, 2)),
    ("Consulta ventas por año/mes", "Con cache", round(tiempo_con_cache, 2))
], ["consulta", "escenario", "tiempo_segundos"])

df_optimizacion.show(truncate=False)

+---------------------------+---------+---------------+
|consulta                   |escenario|tiempo_segundos|
+---------------------------+---------+---------------+
|Consulta ventas por año/mes|Sin cache|0.54           |
|Consulta ventas por año/mes|Con cache|0.31           |
+---------------------------+---------+---------------+



## 2. Data mart: ventas mensuales

In [5]:
from pyspark.sql.functions import (
    col, sum as spark_sum, countDistinct, count, avg, round as spark_round
)

inicio = time.time()

ventas_mensuales = (
    df_silver
    .groupBy("anio", "mes")
    .agg(
        spark_sum("venta").alias("venta_total"),
        spark_sum("costo").alias("costo_total"),
        spark_sum("margen").alias("margen_total"),
        spark_sum("unidades").alias("unidades_total"),
        countDistinct("numero_boleta").alias("boletas_distintas"),
        countDistinct("id_producto").alias("productos_distintos")
    )
    .withColumn("margen_porcentaje", spark_round(col("margen_total") / col("venta_total"), 4))
    .orderBy("anio", "mes")
)

ventas_mensuales.show(20, truncate=False)

(
    ventas_mensuales.write
    .format("delta")
    .mode("overwrite")
    .save(f"{RUTA_GOLD}/ventas_mensuales")
)

print("Tiempo Gold ventas_mensuales:", round(time.time() - inicio, 2), "segundos")


+----+---+-------------+------------+------------+--------------+-----------------+-------------------+-----------------+
|anio|mes|venta_total  |costo_total |margen_total|unidades_total|boletas_distintas|productos_distintos|margen_porcentaje|
+----+---+-------------+------------+------------+--------------+-----------------+-------------------+-----------------+
|2015|1  |3.30037003E8 |1.58452523E8|1.7158448E8 |17625         |1                |12767              |0.5199           |
|2015|2  |3.60394586E8 |1.65354758E8|1.95039828E8|19203         |1                |12759              |0.5412           |
|2015|3  |2.77570498E8 |1.16612189E8|1.60958309E8|11568         |1                |9053               |0.5799           |
|2015|4  |3.10311785E8 |1.17953249E8|1.92358536E8|11406         |1                |8504               |0.6199           |
|2015|5  |5.77872458E8 |2.13104121E8|3.64768337E8|19943         |1                |11798              |0.6312           |
|2015|7  |2.87433305E8 |

## 3. Data mart: ventas por tienda

In [6]:
ventas_por_tienda = (
    df_silver
    .groupBy("cod_tienda_facturacion")
    .agg(
        spark_sum("venta").alias("venta_total"),
        spark_sum("costo").alias("costo_total"),
        spark_sum("margen").alias("margen_total"),
        spark_sum("unidades").alias("unidades_total"),
        countDistinct("numero_boleta").alias("boletas_distintas"),
        countDistinct("id_producto").alias("productos_distintos")
    )
    .withColumn("ticket_promedio", spark_round(col("venta_total") / col("boletas_distintas"), 2))
    .withColumn("margen_porcentaje", spark_round(col("margen_total") / col("venta_total"), 4))
    .orderBy(col("venta_total").desc())
)

ventas_por_tienda.show(20, truncate=False)

(
    ventas_por_tienda.write
    .format("delta")
    .mode("overwrite")
    .save(f"{RUTA_GOLD}/ventas_por_tienda")
)


+----------------------+------------+------------+------------+--------------+-----------------+-------------------+---------------+-----------------+
|cod_tienda_facturacion|venta_total |costo_total |margen_total|unidades_total|boletas_distintas|productos_distintos|ticket_promedio|margen_porcentaje|
+----------------------+------------+------------+------------+--------------+-----------------+-------------------+---------------+-----------------+
|765                   |5.07703968E8|1.67117864E8|3.40586104E8|8933          |5033             |7690               |100875.02      |0.6708           |
|37                    |4.80600505E8|1.90080884E8|2.90519621E8|12469         |5327             |12190              |90219.73       |0.6045           |
|456                   |4.5223406E8 |1.76703786E8|2.75530274E8|12318         |5572             |11725              |81161.89       |0.6093           |
|19                    |4.28772244E8|1.76558901E8|2.52213343E8|15437         |6681            

## 4. Data mart: ventas por producto

In [7]:
ventas_por_producto = (
    df_silver
    .groupBy("id_producto")
    .agg(
        spark_sum("venta").alias("venta_total"),
        spark_sum("costo").alias("costo_total"),
        spark_sum("margen").alias("margen_total"),
        spark_sum("unidades").alias("unidades_total"),
        countDistinct("numero_boleta").alias("boletas_distintas")
    )
    .withColumn("margen_porcentaje", spark_round(col("margen_total") / col("venta_total"), 4))
    .orderBy(col("venta_total").desc())
)

ventas_por_producto.show(20, truncate=False)

(
    ventas_por_producto.write
    .format("delta")
    .mode("overwrite")
    .save(f"{RUTA_GOLD}/ventas_por_producto")
)


+-----------+-----------+-----------+------------+--------------+-----------------+-----------------+
|id_producto|venta_total|costo_total|margen_total|unidades_total|boletas_distintas|margen_porcentaje|
+-----------+-----------+-----------+------------+--------------+-----------------+-----------------+
|2873735    |3.6825254E7|1.5921474E7|2.090378E7  |7977          |3389             |0.5676           |
|4467504    |3.5959212E7|1.1529695E7|2.4429517E7 |1302          |1345             |0.6794           |
|2823441    |3.4146834E7|1.2762524E7|2.138431E7  |3367          |1287             |0.6262           |
|4714381    |2.6474504E7|8808700.0  |1.7665804E7 |1361          |1244             |0.6673           |
|4205730    |2.5800236E7|1.0234248E7|1.5565988E7 |4667          |4654             |0.6033           |
|4205726    |2.4053933E7|8705390.0  |1.5348543E7 |2029          |2032             |0.6381           |
|508079     |2.2677244E7|8338219.0  |1.4339025E7 |4356          |3529             

## 5. Data mart: ventas por canal y tipo de documento

In [8]:
ventas_canal_documento = (
    df_silver
    .groupBy("id_canal", "tipo_documento")
    .agg(
        spark_sum("venta").alias("venta_total"),
        spark_sum("costo").alias("costo_total"),
        spark_sum("margen").alias("margen_total"),
        spark_sum("unidades").alias("unidades_total"),
        countDistinct("numero_boleta").alias("boletas_distintas")
    )
    .withColumn("margen_porcentaje", spark_round(col("margen_total") / col("venta_total"), 4))
    .orderBy("id_canal", "tipo_documento")
)

ventas_canal_documento.show(truncate=False)

(
    ventas_canal_documento.write
    .format("delta")
    .mode("overwrite")
    .save(f"{RUTA_GOLD}/ventas_canal_documento")
)


+--------+--------------+---------------+---------------+---------------+--------------+-----------------+-----------------+
|id_canal|tipo_documento|venta_total    |costo_total    |margen_total   |unidades_total|boletas_distintas|margen_porcentaje|
+--------+--------------+---------------+---------------+---------------+--------------+-----------------+-----------------+
|1       |B             |9.200756804E9  |4.008029635E9  |5.192727169E9  |381433        |1                |0.5644           |
|1       |BE            |4.3986807366E10|1.7802208464E10|2.6184598902E10|1518323       |856480           |0.5953           |
|1       |BM            |6596795.0      |3182114.0      |3414681.0      |314           |50               |0.5176           |
|1       |FE            |7.65268723E8   |3.20088621E8   |4.45180102E8   |19317         |3751             |0.5817           |
|1       |GE            |3757535.0      |1534676.0      |2222859.0      |103           |33               |0.5916           |


## 6. Evidencia de optimización: particionamiento y cache

In [9]:
# Comparación simple de tiempos con y sin cache para una consulta frecuente
consulta_base = df_silver.filter((col("anio") == 2016) & (col("mes") == 3))

inicio = time.time()
resultado_sin_cache = consulta_base.groupBy("cod_tienda_facturacion").agg(spark_sum("venta").alias("venta_total"))
resultado_sin_cache.count()
tiempo_sin_cache = time.time() - inicio

df_silver.cache()
df_silver.count()

inicio = time.time()
resultado_con_cache = df_silver.filter((col("anio") == 2016) & (col("mes") == 3)) \
    .groupBy("cod_tienda_facturacion") \
    .agg(spark_sum("venta").alias("venta_total"))
resultado_con_cache.count()
tiempo_con_cache = time.time() - inicio

print("Tiempo sin cache:", round(tiempo_sin_cache, 2), "segundos")
print("Tiempo con cache:", round(tiempo_con_cache, 2), "segundos")

evidencia = spark.createDataFrame([
    ("consulta_ventas_tienda_marzo_2016", "sin_cache", float(tiempo_sin_cache)),
    ("consulta_ventas_tienda_marzo_2016", "con_cache", float(tiempo_con_cache))
], ["consulta", "escenario", "tiempo_segundos"])

evidencia.show()

(
    evidencia.write
    .format("delta")
    .mode("overwrite")
    .save(f"{RUTA_GOLD}/evidencia_optimizacion")
)


Tiempo sin cache: 1.62 segundos
Tiempo con cache: 0.48 segundos
+--------------------+---------+------------------+
|            consulta|escenario|   tiempo_segundos|
+--------------------+---------+------------------+
|consulta_ventas_t...|sin_cache| 1.616586685180664|
|consulta_ventas_t...|con_cache|0.4808230400085449|
+--------------------+---------+------------------+



## 7. Resumen de datasets Gold generados

In [10]:
datasets_gold = [
    "ventas_mensuales",
    "ventas_por_tienda",
    "ventas_por_producto",
    "ventas_canal_documento",
    "evidencia_optimizacion"
]

for dataset in datasets_gold:
    ruta = f"{RUTA_GOLD}/{dataset}"
    total = spark.read.format("delta").load(ruta).count()
    print(dataset, "->", total, "registros")


ventas_mensuales -> 99 registros
ventas_por_tienda -> 469 registros
ventas_por_producto -> 420699 registros
ventas_canal_documento -> 6 registros
evidencia_optimizacion -> 2 registros


## Evidencia para Spark UI y FinOps

Durante la ejecución del notebook se debe capturar la Spark UI, especialmente la pestaña **Jobs** y **SQL/DataFrame**, para evidenciar la ejecución distribuida del pipeline.

La optimización aplicada corresponde al uso de cache sobre la capa Silver antes de construir los data marts Gold. Se compararon los tiempos de ejecución antes y después de aplicar cache.

Para FinOps, la solución considera un clúster temporal de Dataproc que se enciende solo durante la ventana de procesamiento batch y se apaga al finalizar. Esto reduce costos operacionales frente a una arquitectura persistente.

In [14]:
# ============================================================
# Estimación FinOps simple
# ============================================================

costo_hora_cluster_usd = 1.20   # supuesto referencial académico
duracion_job_horas = 0.25       # 15 minutos
ejecuciones_mes = 30            # ejecución diaria

costo_mensual_estimado = costo_hora_cluster_usd * duracion_job_horas * ejecuciones_mes

print("Costo hora estimado clúster Dataproc USD:", costo_hora_cluster_usd)
print("Duración estimada por ejecución:", duracion_job_horas, "horas")
print("Ejecuciones mensuales:", ejecuciones_mes)
print("Costo mensual estimado USD:", round(costo_mensual_estimado, 2))

Costo hora estimado clúster Dataproc USD: 1.2
Duración estimada por ejecución: 0.25 horas
Ejecuciones mensuales: 30
Costo mensual estimado USD: 9.0
